# Text-Based Visualizations for IUU Fishing Story

This notebook produces Tableau-ready CSVs and standalone charts for **Story Points 8 & 9** (External Validation + Voices from the Fleet).

**Data sources:**
- `text/GDELT/gdelt_iuu_news_clean.csv` — 1,539 IUU fishing news articles (2020–2024)
- `text/EJF/ejf_testimonies.csv` — 111 fisher testimonies
- `text/EJF/ejf_word_freq.csv` — 500 word frequencies from EJF reports
- `text/EJF/ejf_bigram_freq.csv` — 300 bigram frequencies
- `datasets/iuu_fishing_index.csv` — IUU Fishing Index country scores

**Outputs:** Enriched CSVs in `tableau/final/data/` ready for Tableau import, plus PNG exports for the report.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import os
from collections import Counter
import re

# Paths
PROJECT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
TEXT = os.path.join(PROJECT, 'text')
DATASETS = os.path.join(PROJECT, 'datasets')
OUT = os.path.join(PROJECT, 'tableau', 'final', 'data')
IMG = os.path.join(PROJECT, 'tableau', 'final', 'attachments')
os.makedirs(OUT, exist_ok=True)
os.makedirs(IMG, exist_ok=True)

plt.rcParams.update({
    'figure.facecolor': '#0d1b2a',
    'axes.facecolor': '#0d1b2a',
    'axes.edgecolor': '#4a5568',
    'text.color': '#e0e0e0',
    'axes.labelcolor': '#e0e0e0',
    'xtick.color': '#a0aec0',
    'ytick.color': '#a0aec0',
    'font.size': 11,
    'figure.dpi': 150,
})

print(f'Project root: {PROJECT}')
print(f'Output dir:   {OUT}')

## 1. Load All Text Data

In [ ]:
# GDELT
gdelt = pd.read_csv(os.path.join(TEXT, 'GDELT', 'gdelt_iuu_news_clean.csv'))
gdelt['date'] = pd.to_datetime(gdelt['date'], errors='coerce')
gdelt = gdelt.dropna(subset=['date'])
gdelt['quarter'] = gdelt['date'].dt.to_period('Q').astype(str)
print(f'GDELT: {len(gdelt)} articles, {gdelt["year"].nunique()} years')

# EJF
testimonies = pd.read_csv(os.path.join(TEXT, 'EJF', 'ejf_testimonies.csv'))
word_freq = pd.read_csv(os.path.join(TEXT, 'EJF', 'ejf_word_freq.csv'))
bigram_freq = pd.read_csv(os.path.join(TEXT, 'EJF', 'ejf_bigram_freq.csv'))
print(f'EJF: {len(testimonies)} testimonies, {len(word_freq)} words, {len(bigram_freq)} bigrams')

# IUU Index
iuu_idx = pd.read_csv(os.path.join(DATASETS, 'iuu_fishing_index.csv'))
print(f'IUU Index: {len(iuu_idx)} countries')

---
## 2. GDELT News Analysis (Story Point 8: External Validation)

### 2a. News Volume Over Time by Topic

In [ ]:
# Monthly article counts by query group
monthly = gdelt.groupby(['year_month', 'query_name']).size().reset_index(name='article_count')
monthly_total = gdelt.groupby('year_month').size().reset_index(name='total_articles')

fig, ax = plt.subplots(figsize=(14, 5))
for qname, grp in monthly.groupby('query_name'):
    ax.plot(grp['year_month'], grp['article_count'], marker='.', label=qname, alpha=0.8)
ax.set_xlabel('Month')
ax.set_ylabel('Article Count')
ax.set_title('IUU Fishing News Volume by Query Group (GDELT)', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=8)
# Show every 3rd x-tick
ticks = sorted(monthly['year_month'].unique())
ax.set_xticks(ticks[::3])
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
fig.savefig(os.path.join(IMG, 'gdelt_news_volume.png'), bbox_inches='tight')
plt.show()

# Save for Tableau
monthly.to_csv(os.path.join(OUT, 'gdelt_monthly_by_query.csv'), index=False)
print(f'Saved: gdelt_monthly_by_query.csv ({len(monthly)} rows)')

### 2b. Topic Distribution by Year

In [ ]:
# Aggregate topic flags by year
topic_cols = [c for c in gdelt.columns if c.startswith('topic_')]
yearly_topics = gdelt.groupby('year')[topic_cols].sum().reset_index()
yearly_topics = yearly_topics[yearly_topics['year'] <= 2024]  # exclude 2025 (only 5 articles)

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(yearly_topics))
width = 0.11
colors = ['#ff4444', '#ffaa00', '#ff6b9d', '#4fc3f7', '#66bb6a', '#ce93d8', '#ffb74d']
labels = [c.replace('topic_', '').title() for c in topic_cols]

for i, (col, label) in enumerate(zip(topic_cols, labels)):
    ax.bar(x + i * width, yearly_topics[col], width, label=label, color=colors[i], alpha=0.85)

ax.set_xlabel('Year')
ax.set_ylabel('Articles Mentioning Topic')
ax.set_title('Topic Prevalence in IUU News Coverage by Year', fontsize=14, fontweight='bold')
ax.set_xticks(x + width * 3)
ax.set_xticklabels(yearly_topics['year'].astype(int))
ax.legend(fontsize=9, ncol=4)
plt.tight_layout()
fig.savefig(os.path.join(IMG, 'gdelt_topics_by_year.png'), bbox_inches='tight')
plt.show()

yearly_topics.to_csv(os.path.join(OUT, 'gdelt_yearly_topics.csv'), index=False)
print(f'Saved: gdelt_yearly_topics.csv')

### 2c. GDELT Title Word Cloud Data
Extract top words from GDELT titles for a word cloud in Tableau.

In [ ]:
# Tokenize GDELT titles
stopwords = set('the a an and or but in on of to for is it that this was with as at by from are be'
                ' has have had been its not will can do did may would should could about all also'
                ' more than over into their they them some other which what when where who how'
                ' new s t us said says after between most no so up out just these those he she we'
                ' his her our if your my am were being each few many much two first one any'.split())

all_words = []
for title in gdelt['title'].dropna():
    words = re.findall(r'[a-z]+', title.lower())
    all_words.extend([w for w in words if w not in stopwords and len(w) > 2])

word_counts = Counter(all_words).most_common(200)
gdelt_words = pd.DataFrame(word_counts, columns=['word', 'count'])
gdelt_words.to_csv(os.path.join(OUT, 'gdelt_title_word_freq.csv'), index=False)
print(f'Top 20 GDELT title words:')
print(gdelt_words.head(20).to_string(index=False))

### 2d. GDELT Enriched with Domain Country (best-effort)

In [ ]:
# Extract domain from URL
from urllib.parse import urlparse

def extract_domain(url):
    try:
        return urlparse(str(url)).netloc.replace('www.', '')
    except:
        return ''

# Common TLD-to-country mapping for news sources
tld_country = {
    '.uk': 'UK', '.au': 'Australia', '.ca': 'Canada', '.nz': 'New Zealand',
    '.in': 'India', '.sg': 'Singapore', '.ph': 'Philippines', '.ng': 'Nigeria',
    '.za': 'South Africa', '.ke': 'Kenya', '.id': 'Indonesia', '.jp': 'Japan',
    '.kr': 'South Korea', '.cn': 'China', '.tw': 'Taiwan', '.fr': 'France',
    '.de': 'Germany', '.es': 'Spain', '.it': 'Italy', '.no': 'Norway',
    '.se': 'Sweden', '.ie': 'Ireland', '.pt': 'Portugal', '.br': 'Brazil',
    '.ar': 'Argentina', '.cl': 'Chile', '.pe': 'Peru', '.mx': 'Mexico',
}

gdelt['domain'] = gdelt['url'].apply(extract_domain)

def guess_country(domain):
    for tld, country in tld_country.items():
        if domain.endswith(tld):
            return country
    if domain.endswith('.com') or domain.endswith('.org') or domain.endswith('.net'):
        return 'US/International'
    return 'Other'

gdelt['source_country'] = gdelt['domain'].apply(guess_country)

# Save enriched version
gdelt.to_csv(os.path.join(OUT, 'gdelt_enriched.csv'), index=False)

print('Source country distribution (top 15):')
print(gdelt['source_country'].value_counts().head(15).to_string())
print(f'\nSaved: gdelt_enriched.csv ({len(gdelt)} rows)')

---
## 3. EJF Testimony Analysis (Story Point 9: Voices from the Fleet)

### 3a. Testimony Themes by Vessel Flag

In [ ]:
# Filter to rows with known flag
t_flagged = testimonies.dropna(subset=['vessel_flag']).copy()
theme_cols = [c for c in t_flagged.columns if c.startswith('theme_')]

theme_by_flag = t_flagged.groupby('vessel_flag')[theme_cols].sum().T
theme_labels = [c.replace('theme_', '').replace('_', ' ').title() for c in theme_cols]

fig, ax = plt.subplots(figsize=(10, 6))
theme_by_flag.index = theme_labels
theme_by_flag.plot(kind='barh', ax=ax, color=['#ff4444', '#ffaa00', '#4fc3f7'], width=0.7)
ax.set_xlabel('Number of Testimonies Mentioning Theme')
ax.set_title('Testimony Themes by Vessel Flag State', fontsize=14, fontweight='bold')
ax.legend(title='Vessel Flag')
plt.tight_layout()
fig.savefig(os.path.join(IMG, 'ejf_themes_by_flag.png'), bbox_inches='tight')
plt.show()

### 3b. Top Bigrams Bar Chart

In [ ]:
top_bg = bigram_freq.head(25).copy()

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#ff6b6b' if any(w in bg for w in ['forced','abuse','labour','slave','violence'])
          else '#ffaa00' if any(w in bg for w in ['iuu','illegal','poach','dark'])
          else '#4fc3f7' for bg in top_bg['bigram']]
ax.barh(top_bg['bigram'][::-1], top_bg['count'][::-1], color=colors[::-1])
ax.set_xlabel('Frequency')
ax.set_title('Top 25 Bigrams in EJF Fisher Reports', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(IMG, 'ejf_top_bigrams.png'), bbox_inches='tight')
plt.show()

### 3c. Sentiment Distribution by Flag

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
flag_colors = {'Chinese': '#ff4444', 'Taiwanese': '#ffaa00', 'Korean': '#4fc3f7'}
t_sent = t_flagged[t_flagged['word_count'] < 500].copy()  # filter outliers
for flag, grp in t_sent.groupby('vessel_flag'):
    ax.scatter(grp['word_count'], grp['sentiment_score'],
              s=grp['neg_word_count'] * 20 + 30,
              c=flag_colors.get(flag, '#78909c'),
              alpha=0.6, label=flag, edgecolors='white', linewidth=0.5)
ax.axhline(y=0, color='#4a5568', linestyle='--', alpha=0.5)
ax.set_xlabel('Quote Length (words)')
ax.set_ylabel('Sentiment Score')
ax.set_title('Testimony Sentiment vs Length (sized by negative word count)', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
fig.savefig(os.path.join(IMG, 'ejf_sentiment_scatter.png'), bbox_inches='tight')
plt.show()

### 3d. Testimonies Over Time

In [ ]:
yearly_test = t_flagged.groupby(['year', 'vessel_flag']).size().reset_index(name='count')
yearly_test = yearly_test[yearly_test['year'] <= 2025]

fig, ax = plt.subplots(figsize=(10, 5))
for flag, grp in yearly_test.groupby('vessel_flag'):
    ax.bar(grp['year'] + {'Chinese': -0.25, 'Taiwanese': 0, 'Korean': 0.25}.get(flag, 0),
           grp['count'], width=0.25, label=flag,
           color=flag_colors.get(flag, '#78909c'), alpha=0.85)
ax.set_xlabel('Year')
ax.set_ylabel('Number of Testimonies')
ax.set_title('Fisher Testimonies by Year and Vessel Flag', fontsize=14, fontweight='bold')
ax.legend()
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
fig.savefig(os.path.join(IMG, 'ejf_testimonies_by_year.png'), bbox_inches='tight')
plt.show()

---
## 4. Cross-Layer Analysis: GDELT News × GFW Risk

### 4a. China-Mention Trend vs IUU Index Score
Does news attention on China correlate with China's position as the highest-risk flag state?

In [ ]:
# China mention share per year
yearly_china = gdelt[gdelt['year'] <= 2024].groupby('year').agg(
    total=('title', 'size'),
    china_mentions=('topic_china', 'sum')
).reset_index()
yearly_china['china_pct'] = (yearly_china['china_mentions'] / yearly_china['total'] * 100).round(1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: China mention % over time
ax1.bar(yearly_china['year'], yearly_china['china_pct'], color='#ff6b6b', alpha=0.8)
ax1.set_xlabel('Year')
ax1.set_ylabel('% of Articles Mentioning China')
ax1.set_title('China in IUU News Coverage', fontsize=13, fontweight='bold')
ax1.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
for i, row in yearly_china.iterrows():
    ax1.text(row['year'], row['china_pct'] + 0.5, f"{row['china_pct']}%",
            ha='center', fontsize=9, color='#e0e0e0')

# Right: Topic co-occurrence heatmap
topic_cols = [c for c in gdelt.columns if c.startswith('topic_')]
cooc = gdelt[topic_cols].corr()
im = ax2.imshow(cooc.values, cmap='RdYlBu_r', vmin=-0.2, vmax=0.5)
labels = [c.replace('topic_', '') for c in topic_cols]
ax2.set_xticks(range(len(labels)))
ax2.set_yticks(range(len(labels)))
ax2.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
ax2.set_yticklabels(labels, fontsize=9)
ax2.set_title('Topic Co-occurrence', fontsize=13, fontweight='bold')
plt.colorbar(im, ax=ax2, shrink=0.8)

plt.tight_layout()
fig.savefig(os.path.join(IMG, 'gdelt_china_and_cooccurrence.png'), bbox_inches='tight')
plt.show()

### 4b. GDELT Precision Check (sample 20 articles per topic)
Professor feedback: "hand-check a sample and report precision."

In [ ]:
# Generate a sample for hand-checking
precision_samples = []
for topic in topic_cols:
    flagged = gdelt[gdelt[topic] == 1]
    n = min(20, len(flagged))
    sample = flagged.sample(n, random_state=42)[['title', 'url', 'query_name']].copy()
    sample['topic_flag'] = topic.replace('topic_', '')
    sample['correct'] = ''  # Fill in manually: Y or N
    precision_samples.append(sample)

precision_df = pd.concat(precision_samples, ignore_index=True)
precision_df.to_csv(os.path.join(OUT, 'gdelt_precision_check.csv'), index=False)
print(f'Saved precision check: {len(precision_df)} articles to review')
print(f'Per topic:')
print(precision_df.groupby('topic_flag').size().to_string())
print(f'\nSample entries:')
print(precision_df[['topic_flag', 'title']].head(10).to_string(index=False))

---
## 5. Export Final Tableau-Ready CSVs

In [ ]:
# 1. GDELT enriched (with domain, source_country, quarter)
gdelt.to_csv(os.path.join(OUT, 'gdelt_enriched.csv'), index=False)

# 2. GDELT monthly by query
monthly.to_csv(os.path.join(OUT, 'gdelt_monthly_by_query.csv'), index=False)

# 3. GDELT yearly topics
yearly_topics.to_csv(os.path.join(OUT, 'gdelt_yearly_topics.csv'), index=False)

# 4. GDELT title word frequencies
gdelt_words.to_csv(os.path.join(OUT, 'gdelt_title_word_freq.csv'), index=False)

# 5. EJF files (copy to output for co-location)
import shutil
for f in ['ejf_testimonies.csv', 'ejf_word_freq.csv', 'ejf_bigram_freq.csv']:
    src = os.path.join(TEXT, 'EJF', f)
    dst = os.path.join(OUT, f)
    shutil.copy2(src, dst)

# 6. IUU Fishing Index (copy)
shutil.copy2(os.path.join(DATASETS, 'iuu_fishing_index.csv'), os.path.join(OUT, 'iuu_fishing_index.csv'))

print('\n=== Files in output directory ===')
for f in sorted(os.listdir(OUT)):
    size = os.path.getsize(os.path.join(OUT, f))
    print(f'  {f:40s} {size:>10,} bytes')

---
## 6. Summary of What to Build in Tableau

All CSVs are now in `tableau/final/data/`. Import each as a separate data source.

### For Dashboard 7 (External Validation):
| Tableau Sheet | Data Source | Key Fields |
|---|---|---|
| GDELT News Volume | `gdelt_monthly_by_query.csv` | year_month (Cols), article_count (Rows), query_name (Color) |
| GDELT Topics by Year | `gdelt_yearly_topics.csv` | year (Cols), topic columns (stacked bar) |
| GDELT Source Countries | `gdelt_enriched.csv` | source_country (Rows), CNTD(url) (Cols) |
| GDELT Word Cloud | `gdelt_title_word_freq.csv` | word (Label), count (Size+Color) |

### For Dashboard 8 (Voices from the Fleet):
| Tableau Sheet | Data Source | Key Fields |
|---|---|---|
| EJF Word Cloud | `ejf_word_freq.csv` | word (Label), count (Size+Color) |
| EJF Top Bigrams | `ejf_bigram_freq.csv` | bigram (Rows), count (Cols) |
| Themes by Flag | `ejf_testimonies.csv` | vessel_flag (Cols), theme_* via Measure Values |
| Sentiment Scatter | `ejf_testimonies.csv` | word_count (Cols), sentiment_score (Rows), vessel_flag (Color) |
| Testimonies by Year | `ejf_testimonies.csv` | year (Cols), CNTD(quote) (Rows), vessel_flag (Color) |

### For IUU Index Validation (blend with country_summary):
| Tableau Sheet | Data Source | Key Fields |
|---|---|---|
| IUU Index Scatter | `iuu_fishing_index.csv` blended on iso3=flag | iuu_score_2025 vs avg_risk_score |

---
## 7. Image Exports Summary

PNGs saved in `tableau/final/attachments/` for the LaTeX report:
- `gdelt_news_volume.png` — News volume by query group over time
- `gdelt_topics_by_year.png` — Topic prevalence grouped bar
- `gdelt_china_and_cooccurrence.png` — China mention trend + topic correlation
- `ejf_themes_by_flag.png` — Testimony themes by vessel flag
- `ejf_top_bigrams.png` — Top 25 bigrams
- `ejf_sentiment_scatter.png` — Sentiment vs quote length
- `ejf_testimonies_by_year.png` — Testimonies over time